In [ ]:
# Cell 1: TEST - DeepRadar2022 model evaluation
# https://www.kaggle.com/jacobramey
# https://github.com/rameyjm7

import os
import numpy as np
import h5py, scipy.io as sio
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import load_model
from pathlib import Path
import yaml

# ------------------------------
# Resolve notebook directory safely
# ------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent

print(f"Notebook directory: {notebook_dir}")
print(f"Project root: {project_root}")
outputs_dir_50 = project_root / "outputs" / "50_evaluation_comparison"
outputs_dir_50.mkdir(parents=True, exist_ok=True)
print(f"Outputs dir: {outputs_dir_50}")

# ------------------------------
# Load DeepRadar2022 dataset
# ------------------------------
cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    path = Path(local_cfg.get('dataset_root', '/scratch/rameyjm7/datasets')) / 'DeepRadar2022'
else:
    path = Path('/scratch/rameyjm7/datasets/DeepRadar2022')
print(f"Loading DeepRadar2022 from: {path}")

def load_h5(filepath, key):
    with h5py.File(filepath, "r") as f:
        return np.array(f[key], dtype="float32").T

def load_mat(filepath, key):
    d = sio.loadmat(filepath)
    return d[key]

X_test  = load_h5(path / "X_test.mat", "X_test")
Y_test  = load_mat(path / "Y_test.mat", "Y_test")
lbl_test = load_mat(path / "lbl_test.mat", "lbl_test")

# Extract modulation class and SNR
cls_test, snr_test = lbl_test[:,0].astype(int)-1, lbl_test[:,1]

# ------------------------------
# Normalize IQ per sample
# ------------------------------
def normalize_iq(X):
    Xn = np.empty_like(X)
    for i in range(X.shape[0]):
        scale = np.max(np.abs(X[i])) + 1e-12
        Xn[i] = X[i] / scale
    return Xn

X_test = normalize_iq(X_test)

# ------------------------------
# Append SNR as a third channel
# ------------------------------
def append_snr_feature(X, snr):
    X_out = []
    for i in range(X.shape[0]):
        snr_col = np.full((X.shape[1], 1), snr[i] / 20.0)
        X_out.append(np.concatenate([X[i], snr_col], axis=1))
    return np.array(X_out, dtype=np.float32)

X_test = append_snr_feature(X_test, snr_test)

# ------------------------------
# Load pre-trained model
# ------------------------------
model_path = project_root / "models" / "deepradar2022" / "deepradar2022_cnn_bilstm_final.keras"
print(f"Loading model from: {model_path}")
model = load_model(model_path)

# ------------------------------
# Evaluate model
# ------------------------------
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"\n✅ Model Test Accuracy (all SNRs): {acc*100:.2f}%")

# ------------------------------
# Predictions and metrics
# ------------------------------
Y_pred = model.predict(X_test, verbose=0)
y_true = np.argmax(Y_test, axis=1)
y_pred = np.argmax(Y_pred, axis=1)

# ------------------------------
# Confusion Matrix
# ------------------------------
labels = [
    "LFM", "2FSK", "4FSK", "8FSK", "Costas", "2PSK", "4PSK", "8PSK",
    "Barker", "Huffman", "Frank", "P1", "P2", "P3", "P4", "Px",
    "Zadoff-Chu", "T1", "T2", "T3", "T4", "NM", "Noise"
]

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap="Blues", cbar=True,
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (CNN + BiLSTM, DeepRadar2022)")
plt.tight_layout()
plt.show()

print("\nClassification Report (All SNRs):")
print(classification_report(y_true, y_pred, target_names=labels))

# ------------------------------
# Accuracy vs SNR
# ------------------------------
unique_snrs = sorted(np.unique(snr_test))
acc_snr = []
for snr in unique_snrs:
    idx = np.where(snr_test == snr)[0]
    acc_snr.append(accuracy_score(y_true[idx], y_pred[idx]) * 100)

plt.figure(figsize=(8,5))
plt.plot(unique_snrs, acc_snr, 'b-o')
plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("Recognition Accuracy vs SNR (DeepRadar2022 CNN+BiLSTM)")
plt.grid(True)
plt.tight_layout()
plt.show()


# ------------------------------
# Accuracy vs SNR per class (DeepRadar2022)
# ------------------------------
class_ids = np.array(sorted(np.unique(y_true)), dtype=int)
per_class_acc = np.full((len(class_ids), len(unique_snrs)), np.nan, dtype=np.float32)
for i, cls in enumerate(class_ids):
    cls_mask = y_true == cls
    for j, snr in enumerate(unique_snrs):
        idx = np.where((snr_test == snr) & cls_mask)[0]
        if len(idx) > 0:
            per_class_acc[i, j] = accuracy_score(y_true[idx], y_pred[idx]) * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].plot(unique_snrs, acc_snr, marker='o', color='blue')
axes[0].set_title('Recognition Accuracy vs. SNR (DeepRadar2022)')
axes[0].set_xlabel('SNR (dB)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].grid(True, alpha=0.4)

for i, cls in enumerate(class_ids):
    axes[1].plot(unique_snrs, per_class_acc[i], marker='o', linewidth=1.0, label=f'class_{cls}')
axes[1].set_title('Accuracy vs. SNR per Class (DeepRadar2022)')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.4)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7)
plt.tight_layout()

png = outputs_dir_50 / '50_deepradar2022_accuracy_vs_snr_line_plots.png'
plt.savefig(png, dpi=180)
print('Saved line charts:', png)
plt.show()



In [ ]:
# Cell 2: TEST - RML2016 model evaluation
# https://www.kaggle.com/jacobramey
# https://github.com/rameyjm7

import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model
import tensorflow as tf
import os
import yaml

# --------------------------------------------------------------
# Resolve model path
# --------------------------------------------------------------
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
model_path = os.path.join(project_root, "models", "rml2016", "rml2016_lstm_rnn_2024.keras")

print("Resolved model path:", model_path)
assert os.path.exists(model_path), f"Model file not found: {model_path}"

model = load_model(model_path)
print("Model loaded successfully.")
outputs_dir_50 = Path(project_root) / "outputs" / "50_evaluation_comparison"
outputs_dir_50.mkdir(parents=True, exist_ok=True)
print("Outputs dir:", outputs_dir_50)

# --------------------------------------------------------------
# Load RML2016.10a Dataset
# --------------------------------------------------------------
cfg_path = Path(project_root) / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    pkl_path = local_cfg.get('datasets', {}).get('rml2016', {}).get('pkl', '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl')
else:
    pkl_path = '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl'
print("Loading dataset:", pkl_path)

with open(pkl_path, "rb") as f:
    data = pickle.load(f, encoding="latin1")

# --------------------------------------------------------------
# Prepare Data (your exact provided format)
# --------------------------------------------------------------
def prepare_data(data):
    X, y, snrs = [], [], []

    for (mod_type, snr), signals in data.items():
        for signal in signals:
            # IQ: shape (128, 2)
            iq = np.vstack([signal[0], signal[1]]).T

            # SNR feature channel (raw SNR, consistent with your training format)
            snr_col = np.full((128, 1), snr, dtype=np.float32)

            combined = np.hstack([iq, snr_col])  # (128, 3)

            X.append(combined)
            y.append(mod_type)
            snrs.append(snr)   # keep real SNR for analysis

    X = np.array(X)
    y = np.array(y)
    snrs = np.array(snrs)

    # Encode labels
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    # Train/test split
    X_train, X_test, y_train, y_test, snr_train, snr_test = train_test_split(
        X, y_encoded, snrs, test_size=0.2, random_state=42
    )

    # LSTM requires this shape already (128, 3)
    return X_train, X_test, y_train, y_test, snr_train, snr_test, encoder

# Prepare
X_train, X_test, y_train, y_test, snr_train, snr_test, encoder = prepare_data(data)

# --------------------------------------------------------------
# Evaluate model on full test set
# --------------------------------------------------------------
y_pred = np.argmax(model.predict(X_test, verbose=False), axis=1)

# Confusion matrix (ALL SNR)
cm_all = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_all, annot=True, fmt="d", cmap="Blues",
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (All SNR Levels)")
plt.show()

print("\nClassification Report (All SNR Levels):")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

# --------------------------------------------------------------
# Evaluate only SNR > 5 dB subset
# --------------------------------------------------------------
idx_high = np.where(snr_test > 5)[0]

X_high = X_test[idx_high]
y_high = y_test[idx_high]

print(f"\nSamples with SNR > 5 dB: {len(idx_high)}")

y_pred_high = np.argmax(model.predict(X_high, verbose=False), axis=1)

cm_high = confusion_matrix(y_high, y_pred_high)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_high, annot=True, fmt="d", cmap="Blues",
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (SNR > 5 dB)")
plt.show()

print("\nClassification Report (SNR > 5 dB):")
print(classification_report(y_high, y_pred_high, target_names=encoder.classes_))



# --------------------------------------------------------------
# Accuracy vs SNR charts (RML2016)
# --------------------------------------------------------------
unique_snrs = np.array(sorted(np.unique(snr_test)), dtype=int)
overall_acc = []
per_mod_acc = np.full((len(encoder.classes_), len(unique_snrs)), np.nan, dtype=np.float32)

for j, snr in enumerate(unique_snrs):
    idx = np.where(snr_test == snr)[0]
    overall_acc.append(float(np.mean(y_pred[idx] == y_test[idx])) * 100.0)

for c in range(len(encoder.classes_)):
    cls_mask = y_test == c
    for j, snr in enumerate(unique_snrs):
        idx = np.where((snr_test == snr) & cls_mask)[0]
        if len(idx) > 0:
            per_mod_acc[c, j] = float(np.mean(y_pred[idx] == y_test[idx])) * 100.0

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].plot(unique_snrs, overall_acc, marker='o', color='blue')
axes[0].set_title('Recognition Accuracy vs. SNR (RML2016)')
axes[0].set_xlabel('SNR (dB)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].grid(True, alpha=0.4)

for i, mod in enumerate(encoder.classes_):
    axes[1].plot(unique_snrs, per_mod_acc[i], marker='o', linewidth=1.1, label=mod)
axes[1].set_title('Accuracy vs. SNR per Modulation Type (RML2016)')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.4)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
plt.tight_layout()

png = outputs_dir_50 / '50_rml2016_accuracy_vs_snr_line_plots.png'
plt.savefig(png, dpi=180)
print('Saved line charts:', png)
plt.show()



In [ ]:
# Cell 3: TEST - RML2018 model evaluation
# https://www.kaggle.com/jacobramey
# https://github.com/rameyjm7

from pathlib import Path
import ast
import re

import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import json
from tensorflow.keras.models import load_model

# --------------------------------------------------------------
# Resolve paths and config
# --------------------------------------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text())
    dcfg = cfg.get('datasets', {}).get('rml2018', {})
    h5_path = Path(dcfg.get('hdf5', '/scratch/rameyjm7/datasets/RML2018/GOLD_XYZ_OSC.0001_1024.hdf5'))
    classes_path = Path(dcfg.get('classes', '/scratch/rameyjm7/datasets/RML2018/classes.txt'))
    classes_fixed_path = Path(dcfg.get('classes_fixed', '/scratch/rameyjm7/datasets/RML2018/classes-fixed.txt'))
else:
    h5_path = Path('/scratch/rameyjm7/datasets/RML2018/GOLD_XYZ_OSC.0001_1024.hdf5')
    classes_path = Path('/scratch/rameyjm7/datasets/RML2018/classes.txt')
    classes_fixed_path = Path('/scratch/rameyjm7/datasets/RML2018/classes-fixed.txt')

best_ckpt_txt = project_root / 'models' / 'rml2018' / 'checkpoints' / 'best_checkpoint.txt'
default_model_path = project_root / 'models' / 'rml2018' / 'rml2018_lstm_rnn.keras'
if best_ckpt_txt.exists():
    cand = Path(best_ckpt_txt.read_text().strip())
    model_path = cand if cand.exists() else default_model_path
else:
    model_path = default_model_path

print('RML2018 dataset:', h5_path)
print('RML2018 model:', model_path)
outputs_dir_50 = project_root / 'outputs' / '50_evaluation_comparison'
outputs_dir_50.mkdir(parents=True, exist_ok=True)
print('Outputs dir:', outputs_dir_50)

assert h5_path.exists(), f'Missing dataset: {h5_path}'
assert classes_path.exists(), f'Missing classes file: {classes_path}'
assert classes_fixed_path.exists(), f'Missing classes-fixed file: {classes_fixed_path}'
assert model_path.exists(), f'Missing model: {model_path}'

# --------------------------------------------------------------
# Build highest-SNR class-balanced eval split
# --------------------------------------------------------------
def parse_classes(path: Path):
    text = path.read_text()
    match = re.search(r'classes\s*=\s*(\[[\s\S]*?\])', text)
    if not match:
        raise ValueError(f'Could not parse classes from {path}')
    return ast.literal_eval(match.group(1))

classes_orig = parse_classes(classes_path)
classes_fixed = parse_classes(classes_fixed_path)
remap_fixed = np.array([classes_fixed.index(c) for c in classes_orig], dtype=np.int64)

with h5py.File(h5_path, 'r') as h5:
    X = h5['X']
    Y = h5['Y']
    Z = h5['Z']

    snr = Z[:, 0]
    max_snr = int(np.max(snr))
    max_idx = np.where(snr == max_snr)[0]
    y_max = np.argmax(Y[max_idx], axis=1)

    rng = np.random.default_rng(42)
    picked = []
    for cls in np.unique(y_max):
        cls_idx = max_idx[y_max == cls]
        k = min(200, len(cls_idx))
        picked.extend(rng.choice(cls_idx, size=k, replace=False).tolist())

    picked = np.array(sorted(picked), dtype=np.int64)
    x_iq = np.asarray(X[picked], dtype=np.float32)
    y_orig = np.argmax(np.asarray(Y[picked]), axis=1).astype(np.int64)
    snr_vals = np.asarray(Z[picked, 0], dtype=np.float32)

# Build alternate target encodings
y_fixed = remap_fixed[y_orig]
y_names = np.array([classes_orig[i] for i in y_orig])
le = LabelEncoder()
y_le = le.fit_transform(y_names)
classes_le = list(le.classes_)

snr_ch = np.repeat(snr_vals[:, None, None], x_iq.shape[1], axis=1)
X_eval = np.concatenate([x_iq, snr_ch], axis=2).astype(np.float32)

print('X_eval shape:', X_eval.shape)
print('max_snr used:', max_snr)

# --------------------------------------------------------------
# Evaluate model with automatic label-order calibration
# --------------------------------------------------------------
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_eval, verbose=0), axis=1)

acc_orig = accuracy_score(y_orig, y_pred)
acc_fixed = accuracy_score(y_fixed, y_pred)
acc_le = accuracy_score(y_le, y_pred)

candidates = [
    ('classes', y_orig, classes_orig, acc_orig),
    ('classes-fixed', y_fixed, classes_fixed, acc_fixed),
    ('labelencoder', y_le, classes_le, acc_le),
]
order_name, y_true, target_names, acc = max(candidates, key=lambda t: t[3])

print(f'RML2018 mapping calibration: acc_orig={acc_orig:.4f}, acc_fixed={acc_fixed:.4f}, acc_le={acc_le:.4f}')
print('Using mapping:', order_name)
print(f'RML2018 evaluation accuracy: {acc:.4f}')
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(target_names)))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues')
plt.title(f'Confusion Matrix (RML2018, mapping={order_name})')
plt.xlabel('Predicted class index')
plt.ylabel('True class index')
plt.tight_layout()
cm_png = outputs_dir_50 / '50_rml2018_confusion_matrix.png'
plt.savefig(cm_png, dpi=180)
print('Saved confusion matrix:', cm_png)
plt.show()

# Optional: load and plot continuation training curves from outputs/rml2018
history_dir = project_root / 'outputs' / 'rml2018'
history_files = sorted(history_dir.glob('*.history.json'))
if history_files:
    latest_hist = max(history_files, key=lambda p: p.stat().st_mtime)
    print('Using history file:', latest_hist)
    hist = json.loads(latest_hist.read_text())
    acc_h = hist.get('accuracy', [])
    val_acc_h = hist.get('val_accuracy', [])
    loss_h = hist.get('loss', [])
    val_loss_h = hist.get('val_loss', [])

    if acc_h and val_acc_h and loss_h and val_loss_h:
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(acc_h, label='train_accuracy')
        plt.plot(val_acc_h, label='val_accuracy')
        plt.title(f'Continuation Accuracy ({len(acc_h)} epochs)')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.grid(True)
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(loss_h, label='train_loss')
        plt.plot(val_loss_h, label='val_loss')
        plt.title(f'Continuation Loss ({len(loss_h)} epochs)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.legend()

        plt.tight_layout()
        curves_png = outputs_dir_50 / '50_rml2018_continuation_training_curves.png'
        plt.savefig(curves_png, dpi=180)
        print('Saved training curves:', curves_png)
        plt.show()



# --------------------------------------------------------------
# Build broader per-SNR slice and plot line charts (RML2018)
# --------------------------------------------------------------
MAX_PER_CLASS_PER_SNR = 120

X_rows, y_rows, snr_rows = [], [], []
with h5py.File(h5_path, 'r') as h5:
    X_all = h5['X']
    Y_all = h5['Y']
    Z_all = h5['Z']

    buckets = {}
    for i in range(len(X_all)):
        s = int(Z_all[i, 0])
        cls = int(np.argmax(Y_all[i]))
        key = (cls, s)
        buckets.setdefault(key, 0)
        if buckets[key] >= MAX_PER_CLASS_PER_SNR:
            continue
        buckets[key] += 1

        iq = np.asarray(X_all[i], dtype=np.float32)
        snr_col = np.full((iq.shape[0], 1), s, dtype=np.float32)
        X_rows.append(np.concatenate([iq, snr_col], axis=1))
        y_rows.append(cls)
        snr_rows.append(s)

X_snr = np.asarray(X_rows, dtype=np.float32)
y_snr_orig = np.asarray(y_rows, dtype=np.int64)
snr_vals = np.asarray(snr_rows, dtype=np.int64)
y_snr = remap[y_snr_orig]

y_snr_pred = np.argmax(model.predict(X_snr, verbose=0), axis=1)
unique_snrs = np.array(sorted(np.unique(snr_vals)), dtype=int)

overall_acc = []
per_mod_acc = np.full((len(classes_fixed), len(unique_snrs)), np.nan, dtype=np.float32)
for j, s in enumerate(unique_snrs):
    idx = np.where(snr_vals == s)[0]
    overall_acc.append(float(np.mean(y_snr_pred[idx] == y_snr[idx])) * 100.0)

for c in range(len(classes_fixed)):
    cls_mask = y_snr == c
    for j, s in enumerate(unique_snrs):
        idx = np.where((snr_vals == s) & cls_mask)[0]
        if len(idx) > 0:
            per_mod_acc[c, j] = float(np.mean(y_snr_pred[idx] == y_snr[idx])) * 100.0

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].plot(unique_snrs, overall_acc, marker='o', color='blue')
axes[0].set_title('Recognition Accuracy vs. SNR (RML2018)')
axes[0].set_xlabel('SNR (dB)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].grid(True, alpha=0.4)

for i, mod in enumerate(classes_fixed):
    axes[1].plot(unique_snrs, per_mod_acc[i], marker='o', linewidth=1.0, label=mod)
axes[1].set_title('Accuracy vs. SNR per Modulation Type (RML2018)')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.4)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7)
plt.tight_layout()

png = outputs_dir_50 / '50_rml2018_accuracy_vs_snr_line_plots.png'
plt.savefig(png, dpi=180)
print('Saved line charts:', png)
plt.show()


